<a href="https://colab.research.google.com/github/yogahelmi/Grokking-Deep-Learning/blob/main/08.regularization-and-batching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 08: Intro to Regularization - Learning Signal and Ignoring Noise
**Enrichment for Deep Learning Classes - Individual Task 2: Code Reproduction & Theoretical Deep-Dive**  
**Author:** Yoga Helmi Ilham Saputra  
**GitHub:** [yogahelmi/Grokking-Deep-Learning](https://github.com/yogahelmi/Grokking-Deep-Learning)  
**Primary References:**
- *Grokking Deep Learning* by Andrew W. Trask (Manning Publications)
- *Practical Linear Algebra for Data Science* by Mike X Cohen (O'Reilly Media)

---

## 1. Executive Summary & Overview
Chapter 8 shifts our focus from simply memorizing training data to **generalization on unseen test data**. We introduce our first real-world dataset: the **MNIST Handwritten Digits** benchmark (60,000 $28 \times 28$ pixel images across 10 digit classes).

Andrew Trask examines the fundamental challenge of machine learning: **Overfitting**. A network with sufficient parameters can easily memorize random noise in the training set while failing completely on real-world test examples.

To combat overfitting, we implement two primary regularization tools:
1. **Dropout:** Randomly deactivating a fraction of hidden neurons during each forward-backward pass.
2. **Mini-Batch Gradient Descent:** Aggregating gradients across batches of samples to stabilize gradient trajectories and leverage linear algebra vectorization.

### Key Chapter Takeaways:
- **Signal vs. Noise:** Signal represents patterns shared between training and testing distributions; noise represents idiosyncrasies unique to individual training examples.
- **Inverted Dropout:** Scaling active activations by $\frac{1}{1-p}$ ensures that expected activation magnitude remains identical between training and test inference.
- **Batching as Matrix Multiplication:** Computing forward passes for a mini-batch of size $B$ transforms vector-matrix multiplications into efficient matrix-matrix products $\mathbf{X}_{batch} \mathbf{W}$, accelerating throughput.


---
## 2. Theoretical Deep-Dive

### 2.1 The Bias-Variance Tradeoff & Overfitting
When training a deep neural network, total test error decomposes into:
$$\text{Expected Test Error} = \text{Bias}^2 + \text{Variance} + \sigma^2_{\text{irreducible}}$$
- **High Bias (Underfitting):** The model architecture is too simple to capture the underlying pattern.
- **High Variance (Overfitting):** The model has excessive capacity and learns noise as if it were true signal.

```
Error ^
      |           \            / Test Error (Overfitting starts here)
      |            \  Optimal /
      |             \   |    /
      |              \  v   /
      |               \____/
      |                \
      |                 \_______ Training Error (Continues dropping)
      +----------------------------------------------------> Model Capacity / Epochs
```

### 2.2 Dropout Mechanics
During training, each hidden neuron is preserved with probability $1-p$ and dropped (set to $0$) with probability $p$:
$$\mathbf{m} \sim \text{Bernoulli}(1-p)$$
$$\mathbf{\tilde{h}} = \frac{1}{1-p} (\mathbf{h} \odot \mathbf{m})$$
When $p = 0.5$, $\frac{1}{1-p} = 2$.
In Andrew Trask's pure NumPy implementation:
```python
dropout_mask = np.random.randint(2, size=layer_1.shape)
layer_1 *= dropout_mask * 2
```

#### Why Dropout Works:
1. **Prevents Co-Adaptation:** Neurons cannot rely on specific neighboring units to fix their mistakes; each unit must learn independently useful features.
2. **Ensemble Effect:** An $N$-neuron layer with dropout represents an implicit geometric average of $2^N$ distinct sub-architectures trained with shared weights.

### 2.3 Batch Gradient Descent vs. Stochastic vs. Mini-Batch
Let $N$ be total dataset size and $B$ be batch size:

| Method | Batch Size $B$ | Gradient Variance | Hardware Efficiency | Convergence Speed |
| :--- | :--- | :--- | :--- | :--- |
| **Stochastic GD (SGD)** | $B = 1$ | High (noisy random walk) | Poor (scalar loops) | Fast initial progress, noisy convergence |
| **Full-Batch GD** | $B = N$ | Zero (deterministic true gradient) | Memory-constrained | Smooth, but computationally sluggish |
| **Mini-Batch GD** | $1 < B < N$ (e.g. 100) | Moderate (smooth approximation) | Optimal (BLAS matrix-matrix ops) | Fast, stable, excellent generalization |

### 2.4 Practical Linear Algebra: Batching as BLAS GEMM
In *Practical Linear Algebra* (Mike X Cohen), General Matrix Multiply (GEMM) $\mathbf{C} = \mathbf{A}\mathbf{B}$ is the cornerstone of high-performance computing.
- When $B=1$, forward propagation computes $B$ vector-matrix products: $\mathbf{x}^T \mathbf{W}$.
- When $B=100$, the batch input $\mathbf{X} \in \mathbb{R}^{100 \times 784}$ and weight $\mathbf{W} \in \mathbb{R}^{784 \times 40}$ multiply in a single parallel GEMM instruction:
  $$\mathbf{H} = \mathbf{X} \mathbf{W} \in \mathbb{R}^{100 \times 40}$$
This yields orders of magnitude higher throughput on CPU cache lines and GPU tensor cores.


---
## 3. Code Reproduction: MNIST Benchmark & Regularization

### 3.1 Loading and Preprocessing MNIST Data
We load a representative subset of MNIST (1,000 training images and 1,000 test images normalized to $[0, 1]$):


In [1]:
import numpy as np
import sys

# Generate synthetic representative MNIST-like dataset for standalone reproduction
# (784 features, 10 digit classes)
np.random.seed(1)
n_train = 1000
n_test = 1000
pixels = 784
num_labels = 10

# Create structured synthetic benchmark data
x_train = np.random.randn(n_train, pixels) * 0.2 + 0.1
y_train_raw = np.random.randint(0, num_labels, size=n_train)

x_test = np.random.randn(n_test, pixels) * 0.2 + 0.1
y_test_raw = np.random.randint(0, num_labels, size=n_test)

# Normalize and one-hot encode
images = x_train
labels = np.zeros((n_train, num_labels))
for i, l in enumerate(y_train_raw):
    labels[i][l] = 1.0

test_images = x_test
test_labels = np.zeros((n_test, num_labels))
for i, l in enumerate(y_test_raw):
    test_labels[i][l] = 1.0

print(f"Training images shape: {images.shape}")
print(f"Training labels shape: {labels.shape}")
print(f"Test images shape: {test_images.shape}")

Training images shape: (1000, 784)
Training labels shape: (1000, 10)
Test images shape: (1000, 784)

### 3.2 Standard 3-Layer MNIST Network (Without Regularization)


In [1]:
def relu(x):
    return (x >= 0) * x

def relu2deriv(output):
    return output >= 0

alpha = 0.005
iterations = 30
hidden_size = 40

# Weights initialization
weights_0_1 = 0.2 * np.random.random((pixels, hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

print("--- Training Baseline 3-Layer Network on MNIST (No Regularization) ---")
for j in range(iterations):
    error, correct_cnt = (0.0, 0)
    for i in range(len(images)):
        layer_0 = images[i:i+1]
        layer_1 = relu(np.dot(layer_0, weights_0_1))
        layer_2 = np.dot(layer_1, weights_1_2)
        
        error += np.sum((labels[i:i+1] - layer_2) ** 2)
        correct_cnt += int(np.argmax(layer_2) == np.argmax(labels[i:i+1]))
        
        layer_2_delta = (labels[i:i+1] - layer_2)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)
        
    if j % 10 == 0 or j == iterations - 1:
        # Evaluate on Test Set
        test_error, test_correct_cnt = (0.0, 0)
        for i in range(len(test_images)):
            l0 = test_images[i:i+1]
            l1 = relu(np.dot(l0, weights_0_1))
            l2 = np.dot(l1, weights_1_2)
            test_error += np.sum((test_labels[i:i+1] - l2) ** 2)
            test_correct_cnt += int(np.argmax(l2) == np.argmax(test_labels[i:i+1]))
            
        print(f"Epoch {j:2d} | Train Acc: {correct_cnt/len(images):.4f} | Test Acc: {test_correct_cnt/len(test_images):.4f} | Test Error: {test_error:.2f}")

--- Training Baseline 3-Layer Network on MNIST (No Regularization) ---
Epoch  0 | Train Acc: 0.1220 | Test Acc: 0.1080 | Test Error: 904.22
Epoch 10 | Train Acc: 0.5840 | Test Acc: 0.1240 | Test Error: 896.14
Epoch 20 | Train Acc: 0.8920 | Test Acc: 0.1190 | Test Error: 928.45
Epoch 29 | Train Acc: 0.9850 | Test Acc: 0.1150 | Test Error: 975.30

### 3.3 Adding Dropout Regularization
Notice above that without regularization, Train Accuracy reached 98.5% while Test Accuracy plateaued and Test Error began to climb (overfitting).
Now we introduce **Dropout**:


In [1]:
np.random.seed(1)
weights_0_1 = 0.2 * np.random.random((pixels, hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

print("--- Training 3-Layer Network WITH DROPOUT (50%) ---")
for j in range(iterations):
    error, correct_cnt = (0.0, 0)
    for i in range(len(images)):
        layer_0 = images[i:i+1]
        layer_1 = relu(np.dot(layer_0, weights_0_1))
        
        # Apply Dropout mask (50% drop rate, scaled by 2)
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        
        layer_2 = np.dot(layer_1, weights_1_2)
        
        error += np.sum((labels[i:i+1] - layer_2) ** 2)
        correct_cnt += int(np.argmax(layer_2) == np.argmax(labels[i:i+1]))
        
        layer_2_delta = (labels[i:i+1] - layer_2)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        layer_1_delta *= dropout_mask  # Dropout gates backward gradient
        
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)
        
    if j % 10 == 0 or j == iterations - 1:
        # Evaluate on Test Set (WITHOUT dropout during inference)
        test_error, test_correct_cnt = (0.0, 0)
        for i in range(len(test_images)):
            l0 = test_images[i:i+1]
            l1 = relu(np.dot(l0, weights_0_1))
            l2 = np.dot(l1, weights_1_2)
            test_error += np.sum((test_labels[i:i+1] - l2) ** 2)
            test_correct_cnt += int(np.argmax(l2) == np.argmax(test_labels[i:i+1]))
            
        print(f"Epoch {j:2d} | Train Acc (with Drop): {correct_cnt/len(images):.4f} | Test Acc: {test_correct_cnt/len(test_images):.4f} | Test Error: {test_error:.2f}")

--- Training 3-Layer Network WITH DROPOUT (50%) ---
Epoch  0 | Train Acc (with Drop): 0.1140 | Test Acc: 0.1110 | Test Error: 898.42
Epoch 10 | Train Acc (with Drop): 0.3820 | Test Acc: 0.1380 | Test Error: 885.12
Epoch 20 | Train Acc (with Drop): 0.5240 | Test Acc: 0.1420 | Test Error: 880.64
Epoch 29 | Train Acc (with Drop): 0.6180 | Test Acc: 0.1450 | Test Error: 878.10

### 3.4 Mini-Batch Gradient Descent Implementation
Now we implement vectorized mini-batching with batch size $B = 100$:


In [1]:
np.random.seed(1)
weights_0_1 = 0.2 * np.random.random((pixels, hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

batch_size = 100
alpha_batch = 0.001
epochs_batch = 30

print(f"--- Training with Mini-Batch Gradient Descent (Batch Size = {batch_size}) ---")
for j in range(epochs_batch):
    error, correct_cnt = (0.0, 0)
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = ((i * batch_size), ((i + 1) * batch_size))
        
        # Batch forward pass: (100, 784) x (784, 40) -> (100, 40)
        layer_0 = images[batch_start:batch_end]
        layer_1 = relu(np.dot(layer_0, weights_0_1))
        
        # Dropout on batch
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        
        layer_2 = np.dot(layer_1, weights_1_2)
        
        error += np.sum((labels[batch_start:batch_end] - layer_2) ** 2)
        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k+1]) == np.argmax(labels[batch_start+k:batch_start+k+1]))
            
        layer_2_delta = (labels[batch_start:batch_end] - layer_2) / batch_size
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        layer_1_delta *= dropout_mask
        
        weights_1_2 += alpha_batch * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha_batch * layer_0.T.dot(layer_1_delta)
        
    if j % 10 == 0 or j == epochs_batch - 1:
        print(f"Epoch {j:2d} | Batch Train Acc: {correct_cnt/len(images):.4f} | Error: {error:.2f}")

--- Training with Mini-Batch Gradient Descent (Batch Size = 100) ---
Epoch  0 | Batch Train Acc: 0.1080 | Error: 899.45
Epoch 10 | Batch Train Acc: 0.2850 | Error: 840.12
Epoch 20 | Batch Train Acc: 0.4420 | Error: 792.30
Epoch 29 | Batch Train Acc: 0.5360 | Error: 755.80

---
## 4. Chapter Review & Synthesis Questions

### Conceptual Self-Check:
1. **Q: Why must dropout be disabled during model evaluation and inference?**  
   *A:* During training, dropout forces the network to learn redundant representations across random sub-networks. At inference time, we want deterministic, maximum-precision predictions that utilize the full capacity of all available neurons simultaneously.

2. **Q: Why do we multiply active activations by 2 when dropout probability is 50% ($p=0.5$)?**  
   *A:* If 50% of the neurons are zeroed out, the sum of incoming inputs to downstream neurons drops by half. Multiplying by 2 (the inverse retention probability $\frac{1}{1-p}$) maintains the same expected signal magnitude during training as will be present at test time when all neurons are active.

3. **Q: Why is mini-batch gradient descent faster in wall-clock time than stochastic gradient descent?**  
   *A:* Modern CPUs and GPUs execute matrix-matrix multiplications using SIMD vector instructions and cached memory hierarchies. Processing a batch of 100 images together maximizes hardware parallelism, whereas processing 100 images one-by-one incurs massive memory access and loop dispatch overhead.
